In [1]:
print("hi")

hi


In [2]:
pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [4]:
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234",   # change if different
    database="MovieDBN"
)

print("Connection successful!")

Connection successful!


In [5]:
query = "SELECT * FROM movie"

df = pd.read_sql(query, conn)

df.head()

C:\Users\DELL\AppData\Local\Temp\ipykernel_13900\2276660273.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,movie_id,title,year,product_id,duration
0,1,Inception,2010-01-01,2,148
1,2,Avatar,2009-01-01,3,120
2,3,Titanic,1997-01-01,4,120
3,4,Interstellar,2014-01-01,2,169
4,5,Frozen,2013-01-01,5,102


In [6]:
conn.close()

In [7]:
df.head()


,movie_id,title,year,product_id,duration
0,1,Inception,2010-01-01,2,148
1,2,Avatar,2009-01-01,3,120
2,3,Titanic,1997-01-01,4,120
3,4,Interstellar,2014-01-01,2,169
4,5,Frozen,2013-01-01,5,102


In [8]:
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234",   # change if different
    database="retailsalesdb"
)

print("Connection successful!")

Connection successful!


In [9]:
files = [
    r"C:\Users\DELL\Downloads\store_sales_1.csv",
    r"C:\Users\DELL\Downloads\store_sales_2.csv",
    r"C:\Users\DELL\Downloads\store_sales_3.csv"
]

df_list = [pd.read_csv(file) for file in files]
sales_df = pd.concat(df_list, ignore_index=True)

print("Rows Loaded:", len(sales_df))
print(sales_df.head())

Rows Loaded: 300
          ProductName  Qty  Unit_Price    SaleDate CurrencyType  \
0         Smith Paper  3.0        10.5   7/13/2024          OMR   
1      Johnson Screen  NaN         NaN   2/23/2025          Usd   
2  Roberts Ingredient  3.0        30.0  11/13/2024          USD   
3       White Monitor  NaN        10.5   4/16/2025          USD   
4  Rodriguez Keyboard  2.0        20.0    8/3/2024          usd   

                             CustomerID  StoreID  
0  9ca482a2-0356-49c1-b5e3-88ae98d1cc2f  Store_A  
1  c0b9df4e-8f03-4bf0-a31b-0a7d7c2a8907  Store_A  
2  97dc18e3-2c12-4e26-9863-32514e82e822  Store_A  
3  e4d09733-d496-47b3-a4b5-04de84d8fd06  Store_A  
4  435ecb46-4545-4af7-b72c-119f64d193a5  Store_A  


In [ ]:
files = [
    r"C:\Users\DELL\Downloads\store_sales_1.csv",
    r"C:\Users\DELL\Downloads\store_sales_2.csv",
    r"C:\Users\DELL\Downloads\store_sales_3.csv"
]

df_list = [pd.read_csv(file) for file in files]
sales_df = pd.concat(df_list, ignore_index=True)

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd

class SalesCleaner(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        X = X.copy()

        # Missing Values
        X["Qty"] = pd.to_numeric(X["Qty"], errors="coerce")
        X["Unit_Price"] = pd.to_numeric(X["Unit_Price"], errors="coerce")

        X["Qty"] = X["Qty"].fillna(X["Qty"].median())
        X["Unit_Price"] = X["Unit_Price"].fillna(X["Unit_Price"].median())
        X["CustomerID"] = X["CustomerID"].fillna("UNKNOWN")

        # Data Types
        X["Qty"] = X["Qty"].astype(int)
        X["Unit_Price"] = X["Unit_Price"].astype(float)
        X["SaleDate"] = pd.to_datetime(X["SaleDate"])

        # Normalize Text
        text_cols = ["ProductName", "StoreID", "CurrencyType"]

        for col in text_cols:
            X[col] = (
                X[col]
                .astype(str)
                .str.strip()
                .str.upper()
            )

        # Currency Conversion
        USD_TO_OMR = 0.385

        X["Unit_Price_OMR"] = X.apply(
            lambda row:
            row["Unit_Price"] * USD_TO_OMR
            if row["CurrencyType"] == "USD"
            else row["Unit_Price"],
            axis=1
        )

        # New Column
        X["Total_Price"] = X["Qty"] * X["Unit_Price_OMR"]

        # Extra Columns for Easy Searching
        X["Year"] = X["SaleDate"].dt.year
        X["Month"] = X["SaleDate"].dt.month
        X["Day"] = X["SaleDate"].dt.day

        return X

pipeline = Pipeline([
    ("cleaner", SalesCleaner())
])

cleaned_df = pipeline.fit_transform(sales_df)